## Exercise 1: Set up the Pristine Properties Platform

Before diving into data work, you need to establish the Unity Catalog structure and load some property listings. All cells in this exercise are provided — simply run them in order.

### ✅ Provided: Create the Unity Catalog structure

Run the cell below to create:
- A catalog named `realestate_lab`
- A **bronze** schema for raw ingested data
- A **silver** schema for cleansed and transformed data
- A managed volume `raw_files` inside `realestate_lab.bronze`

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
# Use the current catalog to reliably find the workspace default catalog,
# regardless of its naming convention.
default_catalog = spark.catalog.currentCatalog()

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS realestate_lab
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Pristine Properties — real estate analytics platform for Dutch cities'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS realestate_lab.bronze
  COMMENT 'Raw ingested data — as received from agent offices';

CREATE SCHEMA IF NOT EXISTS realestate_lab.silver
  COMMENT 'Cleansed and transformed data — ready for analytics';

CREATE VOLUME IF NOT EXISTS realestate_lab.bronze.raw_files
  COMMENT 'Landing zone for raw CSV files from agent systems';

### ✅ Provided: Populate the volume with sample data

Run the cell below to write four sample CSV files into your volume:
- `property_listings.csv` — 33 property listings with intentional quality issues (duplicates and nulls)
- `real_estate_agents.csv` — 8 agent profiles across 7 Dutch cities
- `property_sales.csv` — 3 completed sale transactions
- `market_stats_wide.csv` — monthly average listing prices per city (wide format)

You will use these files throughout Exercises 2–6.

In [ ]:
# ✅ Run this cell — it writes sample CSV files to your volume

listings_csv = """listing_id,property_id,agent_id,city,property_type,bedrooms,bathrooms,list_price,sqm,listing_date,last_updated,status
1,PROP-001,101,Amsterdam,Apartment,2,1,285000.50,75,2026-01-15,2026-01-15 09:00:00,Active
2,PROP-002,102,Rotterdam,House,4,2,449999.99,145,2026-01-20,2026-01-20 10:30:00,Active
3,PROP-003,101,Amsterdam,Studio,,1,184999.99,42,2026-01-22,2026-01-22 11:00:00,Active
4,PROP-004,103,Utrecht,Villa,5,3,874999.99,280,2026-02-01,2026-02-01 08:45:00,Active
5,PROP-005,102,Rotterdam,Apartment,3,2,320500.25,95,2026-02-05,2026-02-05 14:00:00,Sold
6,PROP-001,101,Amsterdam,Apartment,2,1,279000.00,75,2026-01-15,2026-03-01 09:00:00,Active
7,PROP-006,104,The Hague,House,3,2,395000.00,120,2026-02-10,2026-02-10 16:00:00,Active
8,PROP-007,103,Utrecht,Apartment,1,,165000.00,38,2026-02-14,2026-02-14 11:30:00,Active
9,PROP-008,105,Eindhoven,House,4,3,519999.99,170,2026-02-18,2026-02-18 09:00:00,Active
10,PROP-009,104,The Hague,Studio,1,1,145000.00,35,2026-02-22,2026-02-22 10:00:00,Active
11,PROP-010,101,Amsterdam,Villa,6,4,1249999.99,400,2026-03-01,2026-03-01 14:30:00,Active
12,PROP-002,102,Rotterdam,House,4,2,445000.00,145,2026-01-20,2026-03-05 10:30:00,Active
13,PROP-011,105,Eindhoven,Apartment,2,1,195000.00,68,2026-03-08,2026-03-08 09:00:00,Sold
14,PROP-012,106,Groningen,House,3,,309999.99,105,2026-03-10,2026-03-10 12:00:00,Active
15,PROP-013,102,Rotterdam,Apartment,2,2,265000.00,82,2026-03-15,2026-03-15 15:00:00,Active
16,PROP-014,106,Groningen,Studio,,1,129999.99,33,2026-03-18,2026-03-18 08:30:00,Active
17,PROP-015,103,Utrecht,House,5,3,679999.99,210,2026-03-22,2026-03-22 11:00:00,Active
18,PROP-016,104,The Hague,Apartment,1,1,175000.00,45,2026-03-25,2026-03-25 14:00:00,Sold
19,PROP-017,101,Amsterdam,House,4,2,785000.50,195,2026-04-01,2026-04-01 09:30:00,Active
20,PROP-018,105,Eindhoven,Villa,4,3,649999.99,230,2026-04-05,2026-04-05 10:00:00,Active
21,PROP-019,106,Groningen,Apartment,3,2,215000.00,78,2026-04-08,2026-04-08 11:30:00,Active
22,PROP-020,107,Breda,House,3,2,355000.00,115,2026-04-12,2026-04-12 14:00:00,Active
23,PROP-003,101,Amsterdam,Studio,,1,190000.00,42,2026-01-22,2026-04-15 11:00:00,Active
24,PROP-021,107,Breda,Apartment,2,1,225000.00,72,2026-04-18,2026-04-18 09:00:00,Active
25,PROP-022,102,Rotterdam,Villa,5,,925000.00,310,2026-04-22,2026-04-22 16:00:00,Active
26,PROP-023,103,Utrecht,House,3,2,,105,2026-04-25,2026-04-25 10:30:00,
27,PROP-024,104,The Hague,Apartment,2,1,235000.00,71,2026-05-01,2026-05-01 09:00:00,Active
28,PROP-025,105,Eindhoven,House,,2,410000.00,130,2026-05-05,2026-05-05 11:00:00,Active
29,PROP-026,106,Groningen,Studio,1,1,138000.00,36,2026-05-08,2026-05-08 14:00:00,Active
30,PROP-027,107,Breda,House,4,3,490000.00,158,2026-05-12,2026-05-12 09:30:00,Active
31,PROP-028,101,Amsterdam,Apartment,3,2,345000.00,98,2026-05-15,2026-05-15 10:00:00,Active
32,PROP-029,104,The Hague,House,4,2,,145,2026-05-18,2026-05-18 12:00:00,Active
33,PROP-030,103,Utrecht,Apartment,2,1,245000.00,76,2026-05-22,2026-05-22 15:00:00,Active"""

agents_csv = """agent_id,agent_name,agency,city,phone
101,Sophie van den Berg,Makelaardij De Kust,Amsterdam,+31-20-1234567
102,Jan de Vries,Vastgoed Noord,Rotterdam,+31-10-9876543
103,Maria Hendriks,De Hoek Makelaars,Utrecht,+31-30-4567890
104,Thomas Bakker,Haagse Woning,The Hague,+31-70-2345678
105,Lisa Smits,Brainport Vastgoed,Eindhoven,+31-40-3456789
106,Pieter Jansen,Stadspark Makelaars,Groningen,+31-50-8765432
107,Emma Koster,Markdal Vastgoed,Breda,+31-76-5678901
108,Robert Mulder,Canal Ring Realty,Amsterdam,+31-20-9012345"""

sales_csv = """sale_id,property_id,sale_price,sale_date,days_on_market
1,PROP-005,315000.00,2026-03-15,38
2,PROP-011,192000.00,2026-04-20,43
3,PROP-016,172500.00,2026-04-30,35"""

market_csv = """city,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
Amsterdam,385000,392000,401000,415000,422000,438000,445000,441000,428000,418000,405000,398000
Rotterdam,295000,298000,305000,312000,318000,325000,321000,315000,308000,301000,295000,290000
Utrecht,310000,315000,322000,328000,335000,342000,348000,344000,336000,328000,318000,312000
The Hague,275000,280000,288000,295000,302000,308000,312000,306000,298000,291000,283000,278000
Eindhoven,258000,262000,268000,275000,282000,288000,293000,289000,281000,273000,265000,260000
Groningen,195000,198000,203000,208000,213000,218000,222000,219000,214000,208000,201000,196000
Breda,245000,249000,255000,262000,268000,274000,279000,275000,268000,260000,253000,247000"""

base = "/Volumes/realestate_lab/bronze/raw_files"
dbutils.fs.put(f"{base}/property_listings.csv",   listings_csv, overwrite=True)
dbutils.fs.put(f"{base}/real_estate_agents.csv",  agents_csv,   overwrite=True)
dbutils.fs.put(f"{base}/property_sales.csv",      sales_csv,    overwrite=True)
dbutils.fs.put(f"{base}/market_stats_wide.csv",   market_csv,   overwrite=True)
print("CSV files written to volume.")

### ✅ Provided: Load data into bronze Delta tables

Run the cell below to read all four CSV files from the volume and save them as Delta tables in the `realestate_lab.bronze` schema.

In [ ]:
# ✅ Run this cell — it loads the CSV files into bronze Delta tables

def load_csv(path, table):
    (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
        .write.mode("overwrite")
        .saveAsTable(table))

base = "/Volumes/realestate_lab/bronze/raw_files"
load_csv(f"{base}/property_listings.csv",  "realestate_lab.bronze.listings")
load_csv(f"{base}/real_estate_agents.csv", "realestate_lab.bronze.agents")
load_csv(f"{base}/property_sales.csv",     "realestate_lab.bronze.sales")
load_csv(f"{base}/market_stats_wide.csv",  "realestate_lab.bronze.market_stats")
print("All tables loaded successfully.")

## Exercise 2: Profile the Listings Data

Before transforming data, you need to understand what’s in it. In this exercise, you compute statistics for the `listings` table and generate a summary profile using both SQL and PySpark.

> 💬 **Non-notebook task:** After completing this exercise, follow the steps in the lab setup page to create a visual **Snapshot** profile for `realestate_lab.bronze.listings` in Catalog Explorer.

### Task 1: Compute table statistics

Use `ANALYZE TABLE` to compute statistics for all columns in `realestate_lab.bronze.listings`. Then use `DESCRIBE TABLE EXTENDED` to view the collected metrics.

> 🤖 **Genie Code tip:** Ask Genie Code what statistics `ANALYZE TABLE` collects and how they help the query optimizer.
> *Suggested prompt: "What statistics does ANALYZE TABLE collect in Databricks and how does the query optimizer use them?"*

In [ ]:
%sql
-- Task 1a: Compute statistics for all columns


In [ ]:
%sql
-- Task 1b: View the collected statistics


### Task 2: Generate summary statistics using PySpark

Load the `realestate_lab.bronze.listings` table into a PySpark DataFrame and use `.describe()` to view summary statistics for all columns.

> 🤖 **Genie Code tip:** Ask Genie Code what `.describe()` returns and how it differs from `.summary()`.
> *Suggested prompt: "What is the difference between df.describe() and df.summary() in PySpark?"*

In [ ]:
# Task 2: Load listings and view summary statistics


## Exercise 3: Choosing the Right Data Types

Selecting the correct data type affects data correctness, precision, and performance. You explore three key decisions: `DECIMAL` vs `DOUBLE`, `TIMESTAMP` vs `TIMESTAMP_NTZ`, and the complex types `ARRAY`, `MAP`, and `STRUCT`.

### Task 1: DECIMAL vs DOUBLE for property prices

Floating-point types like `DOUBLE` use binary representation internally, which can introduce small but significant rounding errors — a serious problem for financial data like property prices.

First, run the provided cell to observe this problem. Then write a `CREATE TABLE` statement that stores `list_price` as `DECIMAL(12,2)` by creating `realestate_lab.silver.listings_typed` from the bronze table.

> 🤖 **Genie Code tip:** Ask Genie Code why `DOUBLE` causes precision errors with prices.
> *Suggested prompt: "Why does storing a price like 449999.99 as DOUBLE in SQL lead to precision errors, and what type should I use instead?"*

In [ ]:
%sql
-- ✅ Provided: run this to observe floating-point imprecision with DOUBLE
SELECT
  -- Does 0.1 + 0.2 = 0.3 with DOUBLE?
  CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) as sum_double,
  CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) = CAST(0.3 AS DOUBLE)        AS double_exact,   -- returns FALSE
  CAST(0.1 AS DECIMAL(2,1)) + CAST(0.2 AS DECIMAL(2,1)) as sum_decimal,
  CAST(0.1 AS DECIMAL(2,1)) + CAST(0.2 AS DECIMAL(2,1)) = CAST(0.3 AS DECIMAL(2,1))        AS decimal_exact  -- returns TRUE

In [ ]:
%sql
-- Task 1: Create realestate_lab.silver.listings_typed with CAST for each column
- Hint: CREATE OR REPLACE TABLE ... AS SELECT CAST(...) ...


### Task 2: TIMESTAMP vs TIMESTAMP_NTZ for viewing appointments

When scheduling property viewings, choose the right temporal type:
- `TIMESTAMP`: stores in UTC; displayed adjusted to the session timezone. Use for global events.
- `TIMESTAMP_NTZ`: stores exactly as entered, no conversion. Use for local scheduled appointments.

Run the provided cell to see how the two types behave differently when the session timezone changes.

> 🤖 **Genie Code tip:** Ask Genie Code when to prefer `TIMESTAMP_NTZ` over `TIMESTAMP`.
> *Suggested prompt: "When should I use TIMESTAMP_NTZ instead of TIMESTAMP in Azure Databricks for property viewing appointments?"*

In [ ]:
%sql
-- ✅ Provided: observe how TIMESTAMP and TIMESTAMP_NTZ react to a timezone change
CREATE OR REPLACE TEMP VIEW viewing_appointments AS
SELECT *
FROM VALUES
  ('VA001', 'PROP-001', TIMESTAMP '2026-06-15 09:00:00', TIMESTAMP_NTZ '2026-06-15 09:00:00'),
  ('VA002', 'PROP-005', TIMESTAMP '2026-06-15 14:30:00', TIMESTAMP_NTZ '2026-06-15 14:30:00'),
  ('VA003', 'PROP-010', TIMESTAMP '2026-06-16 10:00:00', TIMESTAMP_NTZ '2026-06-16 10:00:00')
AS t(appointment_id, property_id, viewing_time_ts, viewing_time_ntz);

-- Switch to New York timezone -- TIMESTAMP values shift; TIMESTAMP_NTZ stays unchanged
SET TIME ZONE 'America/New_York';
SELECT appointment_id, property_id, viewing_time_ts, viewing_time_ntz
FROM viewing_appointments;

In [ ]:
%sql
-- ✅ Reset the session timezone back to UTC when done
SET TIME ZONE 'UTC';

### Task 3: Complex types — ARRAY, MAP, and STRUCT

Property listings often carry nested or repeated data: a list of amenities, a set of flexible features, and a structured address. These map naturally to `ARRAY`, `MAP`, and `STRUCT`.

**Task 3a:** Create a temp view called `listings_enriched` with these columns (provide at least 3 rows of sample data using `VALUES`):
- `property_id` STRING
- `amenities` ARRAY\<STRING\> — e.g., `ARRAY('Parking', 'Balcony', 'Storage')`
- `features` MAP\<STRING, STRING\> — e.g., `MAP('energy_label', 'A', 'heating', 'central')`
- `address` STRUCT\<street: STRING, city: STRING, postal_code: STRING\>

**Task 3b:** Query `listings_enriched` and access: the first amenity (index 0), the `energy_label` from features, and the `city` from address.

> 🤖 **Genie Code tip:** Ask how to define and access ARRAY, MAP, and STRUCT in Databricks SQL.
> *Suggested prompt: "How do I create a temp view in Databricks SQL with ARRAY, MAP, and STRUCT columns, and how do I access elements from each?"*

In [ ]:
%sql
-- Task 3a: Create listings_enriched temp view with ARRAY, MAP, and STRUCT columns


In [ ]:
%sql
-- Task 3b: Select from listings_enriched accessing: amenities[0], features['energy_label'], address.city


## Exercise 4: Handling Duplicates and Missing Values

The raw listings data contains duplicate records (the same property re-submitted with updated prices) and missing values in several columns. In this exercise you identify and resolve both issues.

### Task 1: Identify duplicate property listings

Write a SQL query using `GROUP BY` and `HAVING` to find all `property_id` values that appear more than once.

> 🤖 **Genie Code tip:** Ask Genie Code for a SQL pattern to find duplicate values.
> *Suggested prompt: "Write a SQL query to find duplicate property_id values in realestate_lab.bronze.listings."*

In [ ]:
%sql
-- Task 1: Find property_id values that appear more than once
--Hint: GROUP BY property_id HAVING COUNT(*) > 1


### Task 2: Keep the most recently updated listing per property

Use `ROW_NUMBER()` with the `QUALIFY` clause to keep only the most recently updated row for each `property_id`.

> 🤖 **Genie Code tip:** Ask Genie Code about `ROW_NUMBER()` with `QUALIFY` for deduplication.
> *Suggested prompt: "How do I use ROW_NUMBER() with QUALIFY in Databricks SQL to keep only the latest record per property_id ordered by last_updated?"*

In [ ]:
%sql
-- Task 2: Use ROW_NUMBER() with QUALIFY to keep the most recently updated row per property_id
-- Hint: QUALIFY ROW_NUMBER() OVER (PARTITION BY property_id ORDER BY last_updated DESC) = 1


### Task 3: Deduplicate using PySpark

Using PySpark, load the listings table, sort by `last_updated` descending, and call `dropDuplicates(["property_id"])`. Print the count before and after.

> 🤖 **Genie Code tip:** Ask Genie Code how sorting interacts with `dropDuplicates`.
> *Suggested prompt: "How does orderBy work together with dropDuplicates in PySpark to control which duplicate row is kept?"*

In [ ]:
# Task 3: Sort by last_updated DESC and dropDuplicates(['property_id'])
# Print counts before and after, then display the result


### Task 4: Handle null and missing values

The listings table has nulls in several columns. Apply these rules:
- `list_price` is null for withdrawn listings — **drop these rows**
- `bedrooms` is null for some units — **fill with `0`**
- `bathrooms` is null for some properties — **fill with `1.0`**
- `status` is null for incomplete records — **fill with `'Unknown'`**

Use `dropna()` and `fillna()`. Print counts before and after.

> 🤖 **Genie Code tip:** Ask Genie Code how to chain dropna and fillna in PySpark.
> *Suggested prompt: "How do I use dropna and fillna together in PySpark to drop rows where list_price is null and then fill other null columns?"*

In [ ]:
# Task 4: Drop rows where list_price is null, then fillna for bedrooms, bathrooms, status
# Print original and cleaned counts, then display the result


## Exercise 5: Joining Listings with Agents and Sales Data

In this exercise you combine the listings table with agent profiles and sales records using PySpark joins.

### Task 1: Inner join listings with agents

Join `realestate_lab.bronze.listings` with `realestate_lab.bronze.agents` on `agent_id` using an **inner join**. Select: `listing_id`, `property_id`, `city` (from listings), `list_price`, `agent_name`, `agency`.

> 🤖 **Genie Code tip:** Ask Genie Code for the correct PySpark join syntax with column aliases.
> *Suggested prompt: "How do I inner join two PySpark DataFrames on agent_id and select columns from both using DataFrame aliases?"*

In [ ]:
# Task 1: Inner join listings with agents on agent_id
# Select: listing_id, property_id, city (from listings), list_price, agent_name, agency


### Task 2: Find agents with no listings

Perform a **left join** with `agents` on the left and `listings` on the right. Filter for rows where `listing_id` is null to identify agents with no current listings.

> 🤖 **Genie Code tip:** Ask Genie Code how a left join helps find records that exist in one table but not another.
> *Suggested prompt: "How do I use a left join in PySpark to find rows in agents that have no match in listings?"*

In [ ]:
# Task 2: Left join agents (left) with listings (right); filter where listing_id IS NULL
# Select: agent_id, agent_name, agency


### Task 3: Find unsold listings

Join `listings` (left) with `sales` (right) on `property_id` using a **left join**. Filter for rows where `sale_id IS NULL` to find properties that have never been sold.

> 🤖 **Genie Code tip:** Ask Genie Code about the left join + null filter pattern.
> *Suggested prompt: "How do I find rows in the listings table with no matching sale in the sales table using a PySpark left join?"*

In [ ]:
# Task 3: Left join listings (left) with sales (right); filter where sale_id IS NULL
# Select: listing_id, property_id, city, property_type, list_price, listing_date


## Exercise 6: Pivot and Unpivot Market Statistics

Data often arrives in a shape that doesn’t match what analysis requires. In this exercise you rotate rows into columns (pivot) and convert wide-format columns back into rows (unpivot).

### Task 1: Pivot listing counts by property type

Using the bronze listings, write a SQL `PIVOT` query that counts the number of **active** listings per city, with one column per `property_type` (`Apartment`, `House`, `Studio`, `Villa`).

Expected output shape:

| city | Apartment | House | Studio | Villa |
|---|---|---|---|---|
| Amsterdam | 3 | 1 | 1 | 1 |

> 🤖 **Genie Code tip:** Ask Genie Code for the `PIVOT` clause syntax in Databricks SQL.
> *Suggested prompt: "How do I use the PIVOT clause in Databricks SQL to count active listings per city, with one column per property type?"*

In [ ]:
%sql
-- Task 1: Pivot active listing counts by property_type
-- Columns: city, Apartment, House, Studio, Villa



### Task 2: Unpivot the monthly market statistics

The `realestate_lab.bronze.market_stats` table has one row per city and 12 monthly price columns. Use the `UNPIVOT` clause to transform it into long format with columns: `city`, `month`, `avg_price`.

> 🤖 **Genie Code tip:** Ask Genie Code for the UNPIVOT syntax in Databricks SQL.
> *Suggested prompt: "How do I unpivot 12 monthly columns (Jan through Dec) into rows using the UNPIVOT clause in Databricks SQL?"*

In [ ]:
%sql
-- Task 2: Unpivot the 12 monthly columns into rows
-- Expected columns: city, month, avg_price


## Clean up

Run the cell below to remove all lab resources when you are done.

In [ ]:
%sql
-- Uncomment and run to remove all lab resources when you are done
-- DROP CATALOG IF EXISTS realestate_lab CASCADE;